## Import Data

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots


## Clean & Set Up Data

In [4]:
# import .tsv as .csv
df = pd.read_csv('2025_specimen_time_series_events_no_phi.tsv', sep='\t')

In [5]:
df.head()

,accession_id,pat_enc_csn_id,pat_mrn_id,barcode,tube_id,specimen_id,test_code,test_performing_dept,test_performing_location,event_street,event_source,event_type,event_dt
0,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
1,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
2,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,FT4,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
3,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z
4,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z


Reshape to long format (one row per specimen-event) - this way, we can easily compute durations to each event type & compare across specimens

In [6]:
# make sure event_dt is datetime
df['event_dt'] = pd.to_datetime(df['event_dt'])

In [7]:
# pivot to get one column per event type
# if there are multiple events of the same type for a specimen, we take 
# the earliest one (e.g. earliest resulted_dt)
df['test_id'] = df['accession_id'].astype(str) + "_" + df['test_code']

timeline = df.pivot_table(
    index='test_id',
    columns='event_type',
    values='event_dt',
    aggfunc='min'  # in case of duplicates, take earliest
).reset_index()

In [8]:
timeline

event_type,test_id,AdultER,BCSC,BCSC -POCT,BCSC-Heme,Blood Bank,C1,CANHC,CENTRIFUGE,CWS,...,YMBD Remote,cancellation_dt,p512 INFTY,test_collected_dt,test_max_resulted_dt,test_max_verified_dt,test_min_resulted_dt,test_min_verified_dt,test_ordered_dt,test_receipt_dt
0,00007d9883_25HD,NaT,NaT,2025-02-18 12:06:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-02-18 11:59:00+00:00,2025-02-18 16:04:00+00:00,2025-02-18 16:08:00+00:00,2025-02-18 16:04:00+00:00,2025-02-18 16:08:00+00:00,2025-02-18 11:58:00+00:00,2025-02-18 12:06:00+00:00
1,00007d9883_ALSUR,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,2025-02-18 12:02:00+00:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
2,0000b7bd82_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-04-02 11:39:00+00:00,2025-04-02 18:01:00+00:00,2025-04-02 18:15:00+00:00,2025-04-02 18:01:00+00:00,2025-04-02 18:15:00+00:00,2025-04-02 11:38:00+00:00,2025-04-02 17:38:00+00:00
3,0000ec2d4a_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-09-19 14:53:00+00:00,2025-09-20 01:58:00+00:00,2025-09-20 02:04:00+00:00,2025-09-20 01:58:00+00:00,2025-09-20 02:04:00+00:00,2025-09-19 14:52:00+00:00,2025-09-20 01:41:00+00:00
4,00014dfb63_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-06-16 15:20:00+00:00,2025-06-16 16:32:00+00:00,2025-06-16 16:31:00+00:00,2025-06-16 16:32:00+00:00,2025-06-16 16:31:00+00:00,2025-06-16 15:15:00+00:00,2025-06-16 15:50:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249461,fffc9c609a_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-02-05 15:25:00+00:00,2025-02-05 19:56:00+00:00,2025-02-05 20:00:00+00:00,2025-02-05 19:56:00+00:00,2025-02-05 20:00:00+00:00,2025-02-05 15:20:00+00:00,2025-02-05 15:40:00+00:00
249462,fffd4c2fe4_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-12-11 13:34:00+00:00,2025-12-11 16:24:00+00:00,2025-12-11 16:44:00+00:00,2025-12-11 16:24:00+00:00,2025-12-11 16:44:00+00:00,2025-12-11 13:30:00+00:00,2025-12-11 16:09:00+00:00
249463,fffe35a26c_GLUCM,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-09-24 16:03:00+00:00,2025-09-24 16:04:00+00:00,2025-09-24 16:04:00+00:00,2025-09-24 16:03:00+00:00,2025-09-24 16:03:00+00:00,2025-09-24 16:04:00+00:00,2025-09-24 16:03:00+00:00
249464,fffe675962_25HD,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,...,NaT,NaT,NaT,2025-02-06 16:10:00+00:00,2025-02-06 22:36:00+00:00,2025-02-06 22:43:00+00:00,2025-02-06 22:36:00+00:00,2025-02-06 22:43:00+00:00,2025-02-06 16:07:00+00:00,2025-02-06 22:18:00+00:00


In [30]:
timeline.shape

(249466, 67)

In [31]:
# creat test_code for timeline by splitting test_id
timeline['test_code'] = timeline['test_id'].apply(lambda x: x.split('_')[1])

In [47]:
# how many unique test codes in the DF?
unique_test_codes = timeline['test_code'].nunique()
print(f"Number of unique test codes: {unique_test_codes}")

Number of unique test codes: 1022


In [32]:
# Ensure datetime dtype
timeline['test_ordered_dt'] = pd.to_datetime(timeline['test_ordered_dt'], errors='coerce')

# Weekday = 0–4, Weekend = 5–6
timeline['day_of_week'] = timeline['test_ordered_dt'].dt.weekday  # Monday=0
timeline['day_type'] = timeline['day_of_week'].apply(lambda x: 'weekday' if x < 5 else 'weekend')

In [33]:
# what is the relative time of each event compared to the first event (test_ordered_dt) for each test_id?
relative_cols = [
    'test_ordered_dt', 'test_collected_dt', 'test_receipt_dt',
    'test_min_resulted_dt', 'test_max_resulted_dt',
    'test_min_verified_dt', 'test_max_verified_dt'
]

# restore test_ordered_dt from timeline in case it was overwritten in a prior run
timeline['test_ordered_dt'] = timeline['test_id'].map(
    timeline.set_index('test_id')['test_ordered_dt']
)

# ensure datetime dtype before subtraction
for col in relative_cols:
    timeline[col] = pd.to_datetime(timeline[col], utc=True, errors='coerce')

# create new relative columns (do not overwrite original datetime columns)
for col in relative_cols:
    timeline[col + '_relative'] = (
        timeline[col] - timeline['test_ordered_dt']
    ).dt.total_seconds() / 3600  # hours since test_ordered_dt

### Only use "complete" tests (no cancellation data)
Make a copy of the DF, using cols that were not cancelled at any point

In [48]:
# make a df that excludes cancelled orders - we want to look at specimen
# timelines that are "complete" (no cancellations)

# if an order was cancelled, it would have a cancellation_dt, but no resulted_dt or verified_dt
# so only keep the ones where cancellation_dt is null
no_cancel_df = timeline[timeline['cancellation_dt'].isna()].copy()

# no_cancel_df

In [49]:
# timeline - no_cancel --> how many cancelled orders did we remove?
num_rows_removed = timeline.shape[0] - no_cancel_df.shape[0]
print(f"Number of orders removed due to cancellation: {num_rows_removed}")

Number of orders removed due to cancellation: 29063


In [50]:
# no_cancel_df.columns
# no_cancel_df.describe()
# no_cancel_df.head(10)

In [51]:
# get the test code from the test_id column
no_cancel_df['test_code'] = no_cancel_df['test_id'].apply(lambda x: x.split('_')[1])

# no_cancel_df["test_code"]

In [62]:
# Ensure datetime dtype
no_cancel_df['test_ordered_dt'] = pd.to_datetime(no_cancel_df['test_ordered_dt'], errors='coerce')

# Weekday = 0–4, Weekend = 5–6
no_cancel_df['day_of_week'] = no_cancel_df['test_ordered_dt'].dt.weekday  # Monday=0
no_cancel_df['day_type'] = no_cancel_df['day_of_week'].apply(lambda x: 'weekday' if x < 5 else 'weekend')

In [69]:
# num unique test codes in no_cancel_df
unique_test_codes_no_cancel = no_cancel_df['test_code'].nunique()
# print(f"Number of unique test codes in no_cancel_df: {unique_test_codes_no_cancel}")

# get unique test codes in no_cancel_df and how many times they appear (e.g. how many unique test codes and their counts)
test_code_counts = no_cancel_df['test_code'].value_counts().to_dict()

test_code_counts_one_or_more = {test_code: count for test_code, count in test_code_counts.items() if count > 1}
# test_code_counts_one_or_more

Calculate relative time for each specimen/test code journey (e.g. ordered = 0, min_result = 0.14 [happens 0.14 hours after being ordered], etc.)

Order of timeline events:
test_ordered_dt: The date/time the order was placed by the ordering physician.
test_collected_dt: The date/time the specimen was collected from the patient.
test_receipt_dt: The date/time the specimen is logged as received in the lab for testing.
test_min/max_result_dt: The date/time the test was noted as resulted (analysis done).
test_mi/max_verified_dt: The date/time the test results were given to the care team for interpretation and action for the patient's care (process complete).
cancellation: can happen at any time after test_ordered_dt, and is reflected in cancellation_dt.


In [70]:
# what is the relative time of each event compared to the first event (test_ordered_dt) for each test_id?
relative_cols = [
    'test_ordered_dt', 'test_collected_dt', 'test_receipt_dt',
    'test_min_resulted_dt', 'test_max_resulted_dt',
    'test_min_verified_dt', 'test_max_verified_dt'
]

# restore test_ordered_dt from timeline in case it was overwritten in a prior run
no_cancel_df['test_ordered_dt'] = no_cancel_df['test_id'].map(
    timeline.set_index('test_id')['test_ordered_dt']
)

# ensure datetime dtype before subtraction
for col in relative_cols:
    no_cancel_df[col] = pd.to_datetime(no_cancel_df[col], utc=True, errors='coerce')

# create new relative columns (do not overwrite original datetime columns)
for col in relative_cols:
    no_cancel_df[col + '_relative'] = (
        no_cancel_df[col] - no_cancel_df['test_ordered_dt']
    ).dt.total_seconds() / 3600  # hours since test_ordered_dt

In [71]:
# show the new relative columns
relative_cols = [col + '_relative' for col in relative_cols]
# no_cancel_df[relative_cols].head()

Interpreting the relative columns:
- a value of 0 means the event happened at the same time as test_ordered_dt
- apositive value means the event happened after test_ordered_dt
- a negative value would mean the event happened before test_ordered_dt, which shouldn't happen for these events, but could indicate data quality issues if it does

Intepreting a row:
- test_id: 12345_CBC
- test_ordered_dt_relative: 0 (this is the reference point)
- test_collected_dt_relative: 0.083333 (specimen was collected 0.083333 hours after ordering)
- test_receipt_dt_relative: 0.583333 (specimen was received in lab 0.583333 hours after ordering)
- test_min_resulted_dt_relative: 1.283333 (test was resulted 1.283333 hours after ordering)
- test_max_resulted_dt_relative: 1.283333 (same as min resulted in this case)
- test_min_verified_dt_relative: 1.266667 (results were verified 1.266667 hours after ordering) 
- test_max_verified_dt_relative: 1.266667 (same as min_resulted in this case)

All possible locations the test tube could have stopped at (a timestamp in the DF to show a record of the specimen being there)

locations = [
    'AdultER', 'BCSC', 'BCSC -POCT', 'BCSC-Heme', 'Blood Bank',
       'C1', 'CANHC', 'CENTRIFUGE', 'CWS', 'Cancer Cntr', 'CardioV Ctr',
       'Cent-Reflab', 'Chemistry', 'DFLA Remote', 'EAAHC', 'Flow',
       'Hem/Coa/Urn', 'INFTY Rcv', 'IOM_C', 'IOM_H', 'Livonia', 'NCRC HLA',
       'NCRC Immuno', 'NCRC Micro', 'NCRC SpChem', 'NCRC SpQFTB', 'NLNC SP',
       'NOT SENDOUT', 'NoRcv Setup', 'Northv Rmt', 'Northv-poc', 'Northville',
       'P512', 'P612', 'PedsER', 'RIM', 'SENDOUTS', 'SP CoreAuto',
       'Spec. Proc.', 'Taubman1', 'Taubman3', 'UCW', 'UHS', 'West POC',
       'West Recv', 'West Rmote', 'YMBD Remote'
]

In [72]:
# from the data, for each location, find which test codes appear at each location 

locations = [
    'AdultER', 'BCSC', 'BCSC -POCT', 'BCSC-Heme', 'Blood Bank',
       'C1', 'CANHC', 'CENTRIFUGE', 'CWS', 'Cancer Cntr', 'CardioV Ctr',
       'Cent-Reflab', 'Chemistry', 'DFLA Remote', 'EAAHC', 'Flow',
       'Hem/Coa/Urn', 'INFTY Rcv', 'IOM_C', 'IOM_H', 'Livonia', 'NCRC HLA',
       'NCRC Immuno', 'NCRC Micro', 'NCRC SpChem', 'NCRC SpQFTB', 'NLNC SP',
       'NOT SENDOUT', 'NoRcv Setup', 'Northv Rmt', 'Northv-poc', 'Northville',
       'P512', 'P612', 'PedsER', 'RIM', 'SENDOUTS', 'SP CoreAuto',
       'Spec. Proc.', 'Taubman1', 'Taubman3', 'UCW', 'UHS', 'West POC',
       'West Recv', 'West Rmote', 'YMBD Remote'
]

test_codes_by_location = {}

# loop through each location and find the unique test codes that have a timestamp (not null) in that location column (indicating the test went through that location)
for location in locations:

    # if there's a timestamp for that location (i.e. if the test went through that location), then include the test code in the list for that location
    test_codes_by_location[location] = no_cancel_df[no_cancel_df[location].notnull()]['test_code'].unique().tolist()


# get a count of which test codes appear at each location
test_code_counts_by_location = {loc: len(codes) for loc, codes in test_codes_by_location.items()}

In [73]:
# test_code_counts_by_location
# test_codes_by_location

# Visualizations

### Visulization 1 (showing 3???)

Visualize (visual, inspectable, graphic timelines) the “average” time series of events for laboratory orders, given a set of dimensional filters (such as “all CBCs ordered in ward CW1”).  Method is open to interpretation, and whichever means the students find most readily available.

Set up the data (order of timeline events, etc.)

In [21]:
# order of timeline events
event_order = [
    'test_ordered_dt_relative',
    'test_collected_dt_relative',
    'test_receipt_dt_relative',
    'test_min_resulted_dt_relative',
    'test_max_resulted_dt_relative',
    'test_min_verified_dt_relative',
    'test_max_verified_dt_relative'
]

# labels for events
label_map = {
    'test_ordered_dt_relative': 'Ordered',
    'test_collected_dt_relative': 'Collected',
    'test_receipt_dt_relative': 'Received',
    'test_min_resulted_dt_relative': 'Resulted (Start)',
    'test_max_resulted_dt_relative': 'Resulted (End)',
    'test_min_verified_dt_relative': 'Verified (Start)',
    'test_max_verified_dt_relative': 'Verified (End)',
}

In [22]:
# compute average timeline
def compute_average_timeline(df):

    # compute the average time (in hours) for each event across all tests, ignoring nulls
    avg = df[event_order].mean().reset_index()
    
    # rename columns and map event names to labels
    avg.columns = ['event', 'avg_hours']
    
    # map event names to labels, but if an event name is not in the label_map, keep the original name (e.g. for any new events we add later)
    avg['event'] = avg['event'].map(label_map).fillna(avg['event'])

    # return the average timeline as a dataframe with columns "event" and "avg_hours"
    return avg

# get the most common location per event
def annotate_locations(df, event_col, locations):

    # subset the dataframe to only include rows where the event_col is not null (i.e. the test went through that event)
    subset = df[df[event_col].notna()]

    # set up location counts
    loc_counts = {}

    # for each location, count how many tests have a timestamp (not null) in that location column (indicating the test went through that location)
    for loc in locations:
        if loc in subset.columns:
            loc_counts[loc] = subset[loc].notna().sum()  # count tests passing this location
    
    if loc_counts:
        
        # return locations with at least 1 test, separated by comma
        top_locs = [loc for loc, count in loc_counts.items() if count > 0]
        return ', '.join(top_locs) if top_locs else "N/A"
    
    return "N/A"

In [23]:
# plot average timeline with locations shown only on hover
def _location_counts_by_event(df, events, locations):

    # set up a list to hold the records for the location counts by event
    records = []

    # for each event
    for event in events:

        # only get the subset of the DF where the event col is not null (i.e. test wne thru the event)
        subset = df[df[event].notna()]

        # for each location
        for loc in locations:

            # if it's a valid column in the DF
            if loc in df.columns:

                # add a record for this event w/ the locatoin & count of the tests that went through that location for the event
                records.append({
                    "event": label_map.get(event, event), # map event name to label, but if not in label_map, keep original name (e.g. for any new events we add later)
                    "location": loc, # location name
                    "count": subset[loc].notna().sum() # count of tests that went through this location for this event (i.e. count of non-null timestamps in this location column for the subset of tests that went through this event)
                })
    return pd.DataFrame(records)

# plot the avg timeline with locations written out in a separate subplot (bar chart of location counts per event)
def plot_average_timeline(df, test_code, top_n_locations=12):
    if df.empty:
        print(f"No rows found for test_code={test_code}")
        return

    avg_df = compute_average_timeline(df)
    loc_counts_df = _location_counts_by_event(df, event_order, locations)

    # keep only locations that have non-zero activity for this test_code
    location_totals = (
        loc_counts_df.groupby("location", as_index=False)["count"]
        .sum()
        .query("count > 0")
        .sort_values("count", ascending=False)
    )

    # if there are no locations with activity, we can't plot the second subplot, so just show a message and return
    if location_totals.empty:
        print(f"No location activity found for test_code={test_code}")
        return

    # keep top active locations (or all active if fewer)
    location_totals = location_totals.head(top_n_locations)
    all_locations = location_totals["location"].tolist()

    # filter to active locations only
    loc_counts_df = loc_counts_df[loc_counts_df["location"].isin(all_locations)]

    # make subplot title count dynamic based on actual plotted locations
    top_n_locations = len(all_locations)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"Average Timeline ({test_code})",
            f"{top_n_locations} Locations by Event"
        ),
        column_widths=[0.45, 0.55]
    )

    # subplot 1: timeline
    fig.add_trace(
        go.Scatter(
            x=avg_df["avg_hours"], # x-axis is average hours since order
            y=avg_df["event"], # y-axis is event type
            mode="lines+markers", # show both lines and markers
            marker=dict(size=10), 
            line=dict(width=3),
            hovertemplate="Event: %{y}<br>Avg Hours Since Order: %{x:.2f}<extra></extra>" # show event and avg hours on hover, hide trace name
        ),
        row=1, col=1
    )

    # subplot 2: grouped bars of location counts per event
    event_labels = [label_map.get(e, e) for e in event_order]

    # for each event, add a bar trace with counts for each location
    for event_label in event_labels:
        event_slice = loc_counts_df[loc_counts_df["event"] == event_label] # filter to this event
        y_vals = [event_slice.loc[event_slice["location"] == loc, "count"].sum() for loc in all_locations] # get count for each location (0 if not present)

        # add the trace to the figure
        fig.add_trace(
            go.Bar(
                x=all_locations,
                y=y_vals,
                name=event_label
            ),
            row=1, col=2
        )

    # update the layout
    fig.update_layout(
        title=f"Average Timeline + Location Distribution for {test_code}",
        template="plotly_white",
        barmode="group",
        hovermode="closest",
        height=600,
        legend_title="Event"
    )

    # update the axes
    fig.update_yaxes(title_text="Event", autorange="reversed", row=1, col=1)
    fig.update_xaxes(title_text="Hours Since Order", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    fig.update_xaxes(title_text="Location", tickangle=45, row=1, col=2)

    fig.show()

# interactive widget to select test code and plot timeline + location distribution
widgets.interact(
    lambda test_code: plot_average_timeline(
        no_cancel_df[no_cancel_df['test_code'] == test_code], test_code
    ),
    test_code=no_cancel_df['test_code'].unique()
)


interactive(children=(Dropdown(description='test_code', options=('25HD', 'ACTMN', 'CBCD', 'COMP', 'EGLV', 'ETH…

<function __main__.<lambda>(test_code)>

Looks like some test codes have a collection that happens before being recorded as ordered - this could be a data quality issue, or it could be that the collection time is recorded based on when the specimen was collected from the patient, while the order time is based on when the order was entered into the system, and there could be delays in entering orders. we would need to investigate this further by looking at the raw data and understanding the lab workflow. 

##### Average timeline but for cancelled orders ?

In [24]:

# # cancelled orders (cancelled_df will have a cancellation_dt, but may also have other events depending on when the cancellation happened in the process)
# canceled_df = timeline[timeline['cancellation_dt'].notna()].copy()

# # canceled_df.head()

In [25]:
# # do the same thing (subplots as above) but for canceled tests 

# canceled_df = timeline[timeline['cancellation_dt'].notna()].copy()

# # show the cols in canceled_df that have any non-null values (i.e. which events are present in the cancelled orders)
# canceled_df.columns[canceled_df.notna().any()]

# canceled_locations = ['AdultER', 'BCSC -POCT', 'Blood Bank', 'CENTRIFUGE', 'CWS',
#        'Cancer Cntr', 'CardioV Ctr', 'Chemistry', 'Hem/Coa/Urn', 'IOM_C',
#        'IOM_H', 'NCRC HLA', 'NCRC Immuno', 'NCRC Micro', 'NCRC SpChem',
#        'NCRC SpQFTB', 'NLNC SP', 'NOT SENDOUT', 'NoRcv Setup', 'Northv-poc',
#        'P512', 'P612', 'PedsER', 'RIM', 'SENDOUTS', 'SP CoreAuto',
#        'Spec. Proc.', 'Taubman1', 'Taubman3', 'UCW', 'UHS', 'West POC'] #non-null locations in cancelled_df



In [26]:
# # compute average timeline for cancelled tests, and plot with locations as well (similar to the function above but for cancelled_df and canceled_locations instead of no_cancel_df and locations)

# # make a local plotting copy so we don't accidentally mutate earlier cells' objects
# canceled_plot_df = canceled_df.copy()

# # timeline-derived data has test_id but may not have test_code
# if 'test_code' not in canceled_plot_df.columns:
#     canceled_plot_df['test_code'] = canceled_plot_df['test_id'].astype(str).str.split('_').str[-1]

# # ensure relative timeline columns exist for cancelled data
# base_event_cols = [
#     'test_ordered_dt', 'test_collected_dt', 'test_receipt_dt',
#     'test_min_resulted_dt', 'test_max_resulted_dt',
#     'test_min_verified_dt', 'test_max_verified_dt'
# ]

# for col in base_event_cols:
#     if col in canceled_plot_df.columns:
#         canceled_plot_df[col] = pd.to_datetime(canceled_plot_df[col], utc=True, errors='coerce')

# for col in base_event_cols:
#     rel_col = f'{col}_relative'
#     if rel_col not in canceled_plot_df.columns:
#         canceled_plot_df[rel_col] = (
#             canceled_plot_df[col] - canceled_plot_df['test_ordered_dt']
#         ).dt.total_seconds() / 3600

# def plot_average_timeline_cancelled(df, test_code, top_n_locations=12):
#     if df.empty:
#         print(f"No rows found for test_code={test_code}")
#         return

#     df = df.copy()

#     # add cancellation relative time (hours since order) for timeline plotting
#     if (
#         "cancellation_dt_relative" not in df.columns
#         and "cancellation_dt" in df.columns
#         and "test_ordered_dt" in df.columns
#     ):
#         df["cancellation_dt"] = pd.to_datetime(df["cancellation_dt"], utc=True, errors="coerce")
#         df["test_ordered_dt"] = pd.to_datetime(df["test_ordered_dt"], utc=True, errors="coerce")
#         df["cancellation_dt_relative"] = (
#             (df["cancellation_dt"] - df["test_ordered_dt"]).dt.total_seconds() / 3600
#         )

#     # timeline events (include cancellation)
#     timeline_event_order = [
#         "test_ordered_dt_relative",
#         "test_collected_dt_relative",
#         "test_receipt_dt_relative",
#         "test_min_resulted_dt_relative",
#         "test_max_resulted_dt_relative",
#         "test_min_verified_dt_relative",
#         "test_max_verified_dt_relative",
#         "cancellation_dt_relative",
#     ]
#     timeline_event_order = [c for c in timeline_event_order if c in df.columns]

#     local_label_map = {**label_map, "cancellation_dt_relative": "Cancelled"}

#     avg_df = (
#         df[timeline_event_order]
#         .mean(numeric_only=True)
#         .rename_axis("event_col")
#         .reset_index(name="avg_hours")
#     )
#     avg_df["event"] = avg_df["event_col"].map(local_label_map).fillna(avg_df["event_col"])

#     # location activity bars (existing event/location logic)
#     loc_base = canceled_locations if "canceled_locations" in globals() else locations
#     loc_counts_df = _location_counts_by_event(df, event_order, loc_base)

#     location_totals = (
#         loc_counts_df.groupby("location", as_index=False)["count"]
#         .sum()
#         .query("count > 0")
#         .sort_values("count", ascending=False)
#     )
#     if location_totals.empty:
#         print(f"No location activity found for test_code={test_code}")
#         return

#     location_totals = location_totals.head(top_n_locations)
#     all_locations = location_totals["location"].tolist()
#     loc_counts_df = loc_counts_df[loc_counts_df["location"].isin(all_locations)]
#     top_n_locations = len(all_locations)

#     fig = make_subplots(
#         rows=1, cols=2,
#         subplot_titles=(
#             f"Average Timeline ({test_code})",
#             f"{top_n_locations} Locations by Event"
#         ),
#         column_widths=[0.45, 0.55]
#     )

#     # subplot 1: timeline with cancellation included
#     fig.add_trace(
#         go.Scatter(
#             x=avg_df["avg_hours"],
#             y=avg_df["event"],
#             mode="lines+markers",
#             marker=dict(size=10),
#             line=dict(width=3),
#             hovertemplate="Event: %{y}<br>Avg Hours Since Order: %{x:.2f}<extra></extra>"
#         ),
#         row=1, col=1
#     )

#     # subplot 2: grouped bars of location counts per event
#     event_labels = [label_map.get(e, e) for e in event_order]
#     for event_label in event_labels:
#         event_slice = loc_counts_df[loc_counts_df["event"] == event_label]
#         y_vals = [event_slice.loc[event_slice["location"] == loc, "count"].sum() for loc in all_locations]

#         fig.add_trace(
#             go.Bar(x=all_locations, y=y_vals, name=event_label),
#             row=1, col=2
#         )

#     fig.update_layout(
#         title=f"Average Timeline + Location Distribution for Cancelled {test_code}",
#         template="plotly_white",
#         barmode="group",
#         hovermode="closest",
#         height=600,
#         legend_title="Event"
#     )
#     fig.update_yaxes(title_text="Event", autorange="reversed", row=1, col=1)
#     fig.update_xaxes(title_text="Hours Since Order", row=1, col=1)
#     fig.update_yaxes(title_text="Count", row=1, col=2)
#     fig.update_xaxes(title_text="Location", tickangle=45, row=1, col=2)
#     fig.show()


# widgets.interact(
#     lambda test_code: plot_average_timeline_cancelled(
#         canceled_plot_df[canceled_plot_df["test_code"] == test_code], test_code
#     ),
#     test_code=sorted(canceled_plot_df["test_code"].dropna().unique())
# )




### Visualization 2

An A/B version of the above visualization to compare two mutually exclusive datasets (such as “All CBCs ordered in ward CW1, tests ordered on weekdays vs tests ordered on weekends”).

In [74]:
# compute average timeline of 
def compute_average_timeline(df):

    # get the average time (in hours) for each event across all tests, ignoring nulls
    avg = df[event_order].mean().reset_index()

    # avg columns
    avg.columns = ['event', 'avg_hours']

    # map event names to labels, but if an event name is not in the label_map, keep the original name (e.g. for any new events we add later)
    avg['event'] = avg['event'].map(label_map).fillna(avg['event'])

    # return avg DF
    return avg

# count specimens passing through each location per event
def location_counts_by_event(df, events, locations):
    records = [] # store records of event/location/count for each event and location combination

    # for each event
    for event in events:

        # only get the subset of the DF where the event col is not null (i.e. test went thru the event)
        subset = df[df[event].notna()]

        # for each location
        for loc in locations:

            # if it's a valid col in DF
            if loc in df.columns:

                # get count of tests that went through this location for this event 
                # (i.e. count of non-null timestamps in this location column for the subset of tests that went through this event)
                count = subset[loc].sum() if pd.api.types.is_numeric_dtype(subset[loc]) else subset[loc].notna().sum()

                # if it exists
                if count > 0:
                    # add a record for this event w/ the location & count of the tests that went through that location for the event
                    records.append({
                        "event": label_map.get(event, event),
                        "location": loc,
                        "count": count
                    })
    return pd.DataFrame(records)


In [75]:
# filter data for A/B groups (can handle "All") --> by day, by test code
def filter_by_day(df, test_code, location=None, day_type=None):
    filtered = df[df['test_code'] == test_code].copy()

    # filter by location if a specific location is selected
    if location and location != "All":
        filtered = filtered[filtered[location].notna()]

    if day_type:
        if day_type == "weekday":
            filtered = filtered[filtered['test_ordered_dt'].dt.weekday < 5]
        elif day_type == "weekend":
            filtered = filtered[filtered['test_ordered_dt'].dt.weekday >= 5]

    return filtered

# plot the A/B timelines
def plot_ab_timeline(df, test_code, location=None, groupA_name="Group A", groupB_name="Group B"):

    # filter A (e.g., weekdays)
    df_A = filter_by_day(df, test_code, location, day_type="weekday")

    # filter B (e.g., weekends)
    df_B = filter_by_day(df, test_code, location, day_type="weekend")

    # if both are empty, return a message (no plots)
    if df_A.empty and df_B.empty:
        print(f"No data found for test_code={test_code}")
        return

    # compute average timelines for A and B (if not empty)
    avg_A = compute_average_timeline(df_A) if not df_A.empty else None
    avg_B = compute_average_timeline(df_B) if not df_B.empty else None

    # prepare subplots for timelines
    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=(f"Average Timeline: {test_code}"),
        specs=[[{"secondary_y": False}]]
    )

    # Plot A
    if avg_A is not None:
        fig.add_trace(go.Scatter(
            x=avg_A['avg_hours'],
            y=avg_A['event'],
            mode='lines+markers',
            name=groupA_name,
            marker=dict(symbol='circle', size=10),
            line=dict(width=3)
        ))

    # Plot B
    if avg_B is not None:
        fig.add_trace(go.Scatter(
            x=avg_B['avg_hours'],
            y=avg_B['event'],
            mode='lines+markers',
            name=groupB_name,
            marker=dict(symbol='square', size=10),
            line=dict(width=3)
        ))

    fig.update_layout(
        title=f"Average Timeline Comparison ({test_code})",
        xaxis_title="Hours Since Order",
        yaxis_title="Event",
        yaxis=dict(autorange="reversed"),
        template="plotly_white",
        hovermode="closest",
        height=600,
        legend_title="Group"
    )

    fig.show()

# make interactive widget for dashboard-ready comparison
widgets.interact(
    lambda test_code, location: plot_ab_timeline(
        no_cancel_df, test_code, location,
        groupA_name="Weekday", groupB_name="Weekend"
    ),
    test_code=no_cancel_df['test_code'].unique(),
    location=["All"] + locations  # include "All" option at the top
)

interactive(children=(Dropdown(description='test_code', options=('25HD', 'ACTMN', 'CBCD', 'COMP', 'EGLV', 'ETH…

<function __main__.<lambda>(test_code, location)>

Thiis plot allows us to compare the average timelines of tests ordered on weekdays vs weekends, and also see how the distribution of locations involved in the testing process differs between these groups. By selecting a specific test code and optionally filtering by location, we can identify if there are any differences in how quickly tests are processed or which locations are more active for certain test codes on different days of the week.

### Visualization 3

Some way to visualize the likelihood of a named time series event, such as “specimen cancelled due to hemolysis”, in that A/B comparison.

In [76]:
# Visualize likelihood of a named event in an A/B split (weekday vs weekend)

event_col_options = globals().get(
    "event_cols",
    [
        "test_ordered_dt",
        "test_collected_dt",
        "test_receipt_dt",
        "test_min_resulted_dt",
        "test_max_resulted_dt",
        "test_min_verified_dt",
        "test_max_verified_dt",
        "test_cancelled_dt",
    ],
)

# filter the data again (including "All" as an option), 
def filter_data(df, test_code, location="All", day_type=None):
    filtered = df[df["test_code"] == test_code].copy()

    if location and location != "All" and location in filtered.columns:
        filtered = filtered[filtered[location].notna()]

    # if the test_ordered_dt column exists, ensure it's datetime (it should be, but just in case)
    if "test_ordered_dt" in filtered.columns:
        filtered["test_ordered_dt"] = pd.to_datetime(
            filtered["test_ordered_dt"], utc=True, errors="coerce"
        )

    if day_type == "weekday":
        filtered = filtered[filtered["test_ordered_dt"].dt.weekday < 5]
    elif day_type == "weekend":
        filtered = filtered[filtered["test_ordered_dt"].dt.weekday >= 5]

    return filtered

# function to compute the probability of an event occurring (e.g. cancellation) in a given dataframe, based on the presence of a non-null timestamp in the event column,
# return the probability along with the count of occurrences and total count for context
def compute_event_probability(df, event_col):

    # if the event_col is "test_cancelled_dt", we want to check for "cancellation_dt" in the DF since that's the column that indicates cancellation in the data 
    # keep the option open to compute probabilities for other events by using the event_col parameter
    actual_event_col = "cancellation_dt" if event_col == "test_cancelled_dt" else event_col

    # if the DF is empty or the event_col doesn't exist in the DF, we can't compute a probability, so return 0 with counts of 0 for context
    if df.empty or actual_event_col not in df.columns:
        return 0.0, 0, 0

    # total counts is the number of rows in the DF
    # occurred counts is the number of rows where the event_col is not null (indicating the event occurred for that test)
    total = len(df)
    occurred = df[actual_event_col].notna().sum()

    # probability is the count of occurrences divided by total, but if total is 0 (to avoid division by zero), we define the probability as 0
    prob = occurred / total if total > 0 else 0.0
    return prob, occurred, total

In [77]:
# plot the A/B event likelihood as a bar chart with probabilities for each group, and counts in the hover text for context
def plot_ab_event_likelihood(df, test_code, location="All", event_col="cancellation_dt"):
    df_A = filter_data(df, test_code, location, day_type="weekday")
    df_B = filter_data(df, test_code, location, day_type="weekend")


    # compute probabilities and counts for A and B
    prob_A, hit_A, total_A = compute_event_probability(df_A, event_col)
    prob_B, hit_B, total_B = compute_event_probability(df_B, event_col)

    # if both groups have no data (0), return a message instead of plotting
    if total_A == 0 and total_B == 0:
        print(f"No data found for test_code={test_code}, location={location}")
        return

    # make the figure
    fig = go.Figure()

    # plot bar plot --> x is group (weekday vs weekend), y is probability, text is count of occurrences and total for context, hovertemplate shows group, probability, and counts
    fig.add_trace(
        go.Bar(
            x=["Weekday", "Weekend"],
            y=[prob_A, prob_B],
            text=[f"{hit_A}/{total_A} ({prob_A:.1%})", f"{hit_B}/{total_B} ({prob_B:.1%})"],
            textposition="auto",
            marker_color=["steelblue", "darkorange"],
            hovertemplate=(
                "Group: %{x}<br>"
                "Probability: %{y:.2%}<br>"
                "Count: %{text}<extra></extra>"
            ),
        )
    )

    # update layout with dynamic title based on event_col, test_code, and location, and set y-axis to percentage format
    fig.update_layout(
        title=f"Likelihood of '{event_col}' for {test_code} at {location}",
        xaxis_title="Group",
        yaxis_title="Probability",
        yaxis=dict(range=[0, 1], tickformat=".0%"),
        template="plotly_white",
        height=500,
    )

    fig.show()

# make the interactive widget for A/B event likelihood
widgets.interact(
    lambda test_code, location, event_col: plot_ab_event_likelihood(
        timeline, test_code, location, event_col
    ),
    test_code=sorted(timeline["test_code"].dropna().unique()),
    location=["All"] + locations,
    event_col=event_col_options,
)


interactive(children=(Dropdown(description='test_code', options=('$PRVD', '17HP2', '17PRG', '21OH', '23BPT', '…

<function __main__.<lambda>(test_code, location, event_col)>

In [ ]:
# interpretation of a sample result from the A/B event likelihood plot:
# if we see that the probability of cancellation (cancellation_dt) for a specific test code is 5% on weekdays and 10% on weekends at a particular location, with counts of 50/1000 (5%) for weekdays and 30/300 (10%) for weekends, we can interpret this as follows:
# - The likelihood of cancellation for this test code is higher on weekends compared to weekdays (10% vs 5%).
# - The counts provide context: on weekdays, there were 1000 tests with 50 cancellations, while on weekends, there were 300 tests with 30 cancellations.
# - This could indicate that there are operational differences on weekends that lead to a higher cancellation rate, such as reduced staffing, different patient populations, or other factors. Further investigation would be needed to understand the underlying causes and to determine if any interventions are necessary to reduce cancellations on weekends.
# 

This bar chart compares the likelihood of a specific event (e.g., cancellation) occurring for a given test code across two groups: Weekday and Weekend. Each bar represents the probability of the event happening in that group, calculated as the number of occurrences divided by the total number of tests in that group. 
The text on each bar shows the raw count of occurrences over the total tests, along with the percentage. 
A higher bar indicates a greater likelihood of the event occurring in that group, which can help identify if there are significant differences in event rates between weekdays and weekends for that test code.
